# cuslide vs cuslide2 Benchmark

This notebook compares the **original cuslide** plugin against the **cuslide2**
plugin (nvImageCodec-backed) across several workloads and image formats:

| Benchmark | What it measures |
|-----------|------------------|
| Single-tile read | Latency for one 256×256 tile |
| Region read (CPU) | Throughput for a large ROI to host memory |
| Region read (GPU) | Throughput for a large ROI to device memory |
| Multi-tile sequential | N sequential 256×256 reads |
| Batch decode (GPU) | Batched multi-tile decode to GPU |
| Tile-level cache: cold vs warm | Cache hit/miss for identical region re-reads |
| Tile-level cache: aligned vs misaligned | Aligned (start=0) vs misaligned (start=1) sequential tile reads — misaligned patches straddle tile boundaries, so overlapping tiles are reused from cache |
| Tile-level cache: multi-threaded | Concurrent tile reads with ThreadPoolExecutor and caching enabled |

**Test images:**
- Aperio SVS: `/tmp/CMU-1-Small-Region.svs`
- Philips TIFF: `/tmp/Philips-1.tiff`

**Requirements:**
- Both `cucim.kit.cuslide@26.04.00.so` and `cucim.kit.cuslide2@26.04.00.so` built
- CUDA-capable GPU + driver

> **Note:** Each plugin switch requires a kernel restart because cuCIM loads
> plugins once at process startup. This notebook uses `subprocess` to run
> each benchmark in an isolated process.

In [ ]:
import json
import os
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Configuration

In [ ]:
REPO_ROOT = Path(os.path.abspath(".."))
CUSLIDE_LIB = REPO_ROOT / "cpp" / "plugins" / "cucim.kit.cuslide" / "build-release" / "lib"
CUSLIDE2_LIB = REPO_ROOT / "cpp" / "plugins" / "cucim.kit.cuslide2" / "build-release" / "lib"
LIBCUCIM_DIR = REPO_ROOT / "install" / "lib"

PLUGIN_VERSION = "26.04.00"
CUSLIDE_SO = f"cucim.kit.cuslide@{PLUGIN_VERSION}.so"
CUSLIDE2_SO = f"cucim.kit.cuslide2@{PLUGIN_VERSION}.so"

TEST_IMAGES = {
    "Aperio SVS": "/tmp/CMU-1-Small-Region.svs",
    "Philips TIFF": "/tmp/Philips-1.tiff",
}

N_WARMUP = 2
N_REPEATS = 10

assert (CUSLIDE_LIB / CUSLIDE_SO).exists(), f"cuslide plugin not found: {CUSLIDE_LIB / CUSLIDE_SO}"
assert (CUSLIDE2_LIB / CUSLIDE2_SO).exists(), f"cuslide2 plugin not found: {CUSLIDE2_LIB / CUSLIDE2_SO}"
print("Plugin prerequisites OK")
print(f"  cuslide:  {CUSLIDE_LIB / CUSLIDE_SO}")
print(f"  cuslide2: {CUSLIDE2_LIB / CUSLIDE2_SO}")

for name, path in TEST_IMAGES.items():
    exists = Path(path).exists()
    status = "OK" if exists else "MISSING"
    print(f"  {name}: {path} [{status}]")
    if not exists:
        print(f"    WARNING: {name} not found -- its benchmarks will be skipped")

## Benchmark Runner

Since cuCIM loads plugins once per process, we run each benchmark in a
subprocess with the appropriate environment variables.

In [ ]:
BENCHMARK_SCRIPT = textwrap.dedent(r'''
import json
import os
import sys
import time

import numpy as np

plugin_root = os.environ["_BENCH_PLUGIN_ROOT"]
plugin_name = os.environ["_BENCH_PLUGIN_NAME"]
image_path = os.environ["_BENCH_IMAGE"]
n_warmup = int(os.environ.get("_BENCH_WARMUP", "2"))
n_repeats = int(os.environ.get("_BENCH_REPEATS", "10"))

os.environ["CUCIM_PLUGINS"] = plugin_name
os.environ.pop("CUCIM_CONFIG_PATH", None)
os.environ.pop("ENABLE_CUSLIDE2", None)

from cucim.clara import CuImage, _set_plugin_root
_set_plugin_root(plugin_root)

img = CuImage(image_path)
dims = img.resolutions["level_dimensions"]
w, h = dims[0]
tile_sizes = img.resolutions.get("level_tile_sizes", ())
tile_w = tile_sizes[0][0] if tile_sizes and tile_sizes[0][0] > 0 else 256
tile_h = tile_sizes[0][1] if tile_sizes and tile_sizes[0][1] > 0 else 256

results = {}

def bench(name, fn, warmup=n_warmup, repeats=n_repeats):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    times_ms = [t * 1000 for t in times]
    results[name] = {
        "mean_ms": float(np.mean(times_ms)),
        "std_ms": float(np.std(times_ms)),
        "min_ms": float(np.min(times_ms)),
        "max_ms": float(np.max(times_ms)),
        "median_ms": float(np.median(times_ms)),
        "n": repeats,
    }

# --- 1. Single tile read (CPU) ---
bench("single_tile_cpu", lambda: np.asarray(img.read_region((0, 0), (256, 256), level=0)))

# --- 2. Large region read (CPU) ---
roi_w = min(2048, w)
roi_h = min(2048, h)
bench("region_2k_cpu", lambda: np.asarray(img.read_region((0, 0), (roi_w, roi_h), level=0)))

# --- 3. Large region read (GPU) ---
try:
    import cupy as cp
    def gpu_region():
        r = img.read_region((0, 0), (roi_w, roi_h), level=0, device="cuda")
        cp.cuda.get_current_stream().synchronize()
        return r
    bench("region_2k_gpu", gpu_region)
except Exception as e:
    results["region_2k_gpu"] = {"error": str(e)}

# --- 4. Multi-tile sequential (CPU) ---
tile_locs = []
for y in range(0, min(h, tile_h * 4), tile_h):
    for x in range(0, min(w, tile_w * 4), tile_w):
        tile_locs.append((x, y))
n_tiles = len(tile_locs)

def seq_tiles():
    for loc in tile_locs:
        np.asarray(img.read_region(loc, (tile_w, tile_h), level=0))

bench("sequential_tiles_cpu", seq_tiles)
results["sequential_tiles_cpu"]["n_tiles"] = n_tiles

# --- 5. Batch decode (GPU) ---
try:
    import cupy as cp
    batch_locs = tile_locs[:min(16, n_tiles)]
    def batch_gpu():
        tiles = list(img.read_region(
            location=batch_locs,
            size=(tile_w, tile_h),
            level=0,
            device="cuda",
            batch_size=len(batch_locs),
            num_workers=1,
        ))
        cp.cuda.get_current_stream().synchronize()
        return tiles
    bench("batch_tiles_gpu", batch_gpu)
    results["batch_tiles_gpu"]["batch_size"] = len(batch_locs)
except Exception as e:
    results["batch_tiles_gpu"] = {"error": str(e)}

# --- 6. Tile-level cache (both cuslide and cuslide2) ---
try:
    # 6a. Cold vs warm: read the same region twice to measure cache hit speedup
    CuImage.cache("per_process", memory_capacity=256)
    CuImage.cache().record(True)
    img_cached = CuImage(image_path)

    t0 = time.perf_counter()
    np.asarray(img_cached.read_region((0, 0), (roi_w, roi_h), level=0))
    cold_ms = (time.perf_counter() - t0) * 1000
    cold_misses = CuImage.cache().miss_count

    warm_times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        np.asarray(img_cached.read_region((0, 0), (roi_w, roi_h), level=0))
        warm_times.append((time.perf_counter() - t0) * 1000)
    warm_hits = CuImage.cache().hit_count

    results["cache_cold_ms"] = cold_ms
    results["cache_warm_mean_ms"] = float(np.mean(warm_times))
    results["cache_warm_std_ms"] = float(np.std(warm_times))
    results["cache_speedup"] = cold_ms / float(np.mean(warm_times)) if np.mean(warm_times) > 0 else 0
    results["cache_total_hits"] = warm_hits
    results["cache_total_misses"] = cold_misses

    # 6b. Aligned (start=0) vs misaligned (start=1) sequential tile reads
    # Methodology from Multi-thread_and_Multi-process_Tests.ipynb:
    #   Aligned:    patches map 1:1 to tiles, no overlap -> 0 cache hits
    #   Misaligned: patches straddle tile boundaries -> overlapping tiles reused from cache
    grid_extent_x = min(w, tile_w * 8)
    grid_extent_y = min(h, tile_h * 8)

    for start_loc, label in [(0, "aligned"), (1, "misaligned")]:
        CuImage.cache("per_process", memory_capacity=256)
        CuImage.cache().record(True)
        img_grid = CuImage(image_path)

        grid_locs = [(x, y)
                     for y in range(start_loc, grid_extent_y, tile_h)
                     for x in range(start_loc, grid_extent_x, tile_w)]

        t0 = time.perf_counter()
        for loc in grid_locs:
            np.asarray(img_grid.read_region(loc, (tile_w, tile_h), level=0))
        elapsed = (time.perf_counter() - t0) * 1000

        results[f"cache_{label}_ms"] = elapsed
        results[f"cache_{label}_n_patches"] = len(grid_locs)
        results[f"cache_{label}_hits"] = CuImage.cache().hit_count
        results[f"cache_{label}_misses"] = CuImage.cache().miss_count

    # 6c. Multi-threaded tile reads with caching (misaligned, concurrent cache access)
    import concurrent.futures
    CuImage.cache("per_process", memory_capacity=256)
    CuImage.cache().record(True)
    img_mt = CuImage(image_path)

    mt_locs = [(x, y)
               for y in range(1, grid_extent_y, tile_h)
               for x in range(1, grid_extent_x, tile_w)]

    def _read_tile(loc):
        np.asarray(img_mt.read_region(loc, (tile_w, tile_h), level=0))

    n_cache_workers = min(4, os.cpu_count() or 1)
    t0 = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_cache_workers) as executor:
        list(executor.map(_read_tile, mt_locs))
    mt_ms = (time.perf_counter() - t0) * 1000

    results["cache_mt_ms"] = mt_ms
    results["cache_mt_n_patches"] = len(mt_locs)
    results["cache_mt_hits"] = CuImage.cache().hit_count
    results["cache_mt_misses"] = CuImage.cache().miss_count
    results["cache_mt_workers"] = n_cache_workers

except Exception as e:
    results["cache_error"] = str(e)

results["_meta"] = {
    "plugin": plugin_name,
    "image": image_path,
    "image_dims": [w, h],
    "tile_dims": [tile_w, tile_h],
}

print(json.dumps(results))
''')


def run_benchmark(plugin_root, plugin_name, image_path, n_warmup=N_WARMUP, n_repeats=N_REPEATS):
    """Run the benchmark script in a subprocess with the given plugin."""
    env = os.environ.copy()
    env["_BENCH_PLUGIN_ROOT"] = str(plugin_root)
    env["_BENCH_PLUGIN_NAME"] = plugin_name
    env["_BENCH_IMAGE"] = str(image_path)
    env["_BENCH_WARMUP"] = str(n_warmup)
    env["_BENCH_REPEATS"] = str(n_repeats)
    env.pop("CUCIM_PLUGINS", None)
    env.pop("CUCIM_CONFIG_PATH", None)
    env.pop("ENABLE_CUSLIDE2", None)

    ld_path = env.get("LD_LIBRARY_PATH", "")
    libcucim = str(LIBCUCIM_DIR)
    if libcucim not in ld_path:
        env["LD_LIBRARY_PATH"] = f"{libcucim}:{ld_path}" if ld_path else libcucim

    result = subprocess.run(
        [sys.executable, "-c", BENCHMARK_SCRIPT],
        capture_output=True,
        text=True,
        env=env,
        timeout=300,
    )

    if result.returncode != 0:
        print(f"STDERR ({plugin_name}):\n{result.stderr}")
        raise RuntimeError(f"Benchmark failed for {plugin_name}")

    # The last line of stdout is the JSON results
    lines = result.stdout.strip().split("\n")
    return json.loads(lines[-1])

print("Benchmark runner ready")

## Run Benchmarks

In [ ]:
all_results = {}

for image_label, image_path in TEST_IMAGES.items():
    if not Path(image_path).exists():
        print(f"Skipping {image_label} (file not found: {image_path})")
        continue

    print(f"\n{'='*60}")
    print(f"  {image_label}: {image_path}")
    print(f"{'='*60}")

    print(f"  Running cuslide benchmark...")
    cs = run_benchmark(CUSLIDE_LIB, CUSLIDE_SO, image_path)
    print(f"    Done. dims={cs['_meta']['image_dims']}, tiles={cs['_meta']['tile_dims']}")

    print(f"  Running cuslide2 benchmark...")
    cs2 = run_benchmark(CUSLIDE2_LIB, CUSLIDE2_SO, image_path)
    print(f"    Done. dims={cs2['_meta']['image_dims']}, tiles={cs2['_meta']['tile_dims']}")

    all_results[image_label] = {"cuslide": cs, "cuslide2": cs2}

print(f"\nAll benchmarks complete! ({len(all_results)} image(s))")

## Results Table

In [ ]:
BENCH_NAMES = ["single_tile_cpu", "region_2k_cpu", "region_2k_gpu",
               "sequential_tiles_cpu", "batch_tiles_gpu"]

for image_label, res in all_results.items():
    cuslide_results = res["cuslide"]
    cuslide2_results = res["cuslide2"]

    print(f"\n{'='*60}")
    print(f"  {image_label}")
    print(f"  dims={cuslide_results['_meta']['image_dims']}, "
          f"tiles={cuslide_results['_meta']['tile_dims']}")
    print(f"{'='*60}")

    rows = []
    for bm in BENCH_NAMES:
        cs = cuslide_results.get(bm, {})
        cs2 = cuslide2_results.get(bm, {})

        if "error" in cs or "error" in cs2:
            rows.append({
                "Benchmark": bm,
                "cuslide (ms)": cs.get("error", f"{cs.get('mean_ms', 0):.2f} +/- {cs.get('std_ms', 0):.2f}"),
                "cuslide2 (ms)": cs2.get("error", f"{cs2.get('mean_ms', 0):.2f} +/- {cs2.get('std_ms', 0):.2f}"),
                "Speedup": "N/A",
            })
            continue

        cs_mean = cs.get("mean_ms", 0)
        cs2_mean = cs2.get("mean_ms", 0)
        speedup = cs_mean / cs2_mean if cs2_mean > 0 else float("inf")

        extra = ""
        if "n_tiles" in cs:
            extra = f" ({cs['n_tiles']} tiles)"
        if "batch_size" in cs:
            extra = f" (batch={cs['batch_size']})"

        rows.append({
            "Benchmark": bm + extra,
            "cuslide (ms)": f"{cs_mean:.2f} +/- {cs.get('std_ms', 0):.2f}",
            "cuslide2 (ms)": f"{cs2_mean:.2f} +/- {cs2.get('std_ms', 0):.2f}",
            "Speedup": f"{speedup:.2f}x",
        })

    df = pd.DataFrame(rows)
    display(df)

    has_cs_cache = "cache_cold_ms" in cuslide_results
    has_cs2_cache = "cache_cold_ms" in cuslide2_results
    if has_cs_cache or has_cs2_cache:
        print(f"\n  --- Tile-level Cache: Cold vs Warm ---")
        cache_rows = []
        for label, r in [("cuslide", cuslide_results), ("cuslide2", cuslide2_results)]:
            if "cache_cold_ms" in r:
                cache_rows.append({
                    "Plugin": label,
                    "Cold (ms)": f"{r['cache_cold_ms']:.2f}",
                    "Warm (ms)": f"{r['cache_warm_mean_ms']:.2f} +/- {r['cache_warm_std_ms']:.2f}",
                    "Speedup": f"{r['cache_speedup']:.2f}x",
                    "Hits": r["cache_total_hits"],
                    "Misses": r["cache_total_misses"],
                })
            elif "cache_error" in r:
                cache_rows.append({"Plugin": label, "Cold (ms)": f"error: {r['cache_error']}"})
        display(pd.DataFrame(cache_rows))

    has_cs_aligned = "cache_aligned_ms" in cuslide_results
    has_cs2_aligned = "cache_aligned_ms" in cuslide2_results
    if has_cs_aligned or has_cs2_aligned:
        print(f"\n  --- Tile-level Cache: Aligned vs Misaligned Sequential Reads ---")
        print(f"  (Aligned=patches on tile grid, Misaligned=patches straddle boundaries)")
        align_rows = []
        for label, r in [("cuslide", cuslide_results), ("cuslide2", cuslide2_results)]:
            if "cache_aligned_ms" in r:
                align_rows.append({
                    "Plugin": label,
                    "Aligned (ms)": f"{r['cache_aligned_ms']:.2f}",
                    "Aligned hits/misses": f"{r['cache_aligned_hits']}/{r['cache_aligned_misses']}",
                    "Misaligned (ms)": f"{r['cache_misaligned_ms']:.2f}",
                    "Misaligned hits/misses": f"{r['cache_misaligned_hits']}/{r['cache_misaligned_misses']}",
                    "Patches": r["cache_aligned_n_patches"],
                })
        display(pd.DataFrame(align_rows))

    has_cs_mt = "cache_mt_ms" in cuslide_results
    has_cs2_mt = "cache_mt_ms" in cuslide2_results
    if has_cs_mt or has_cs2_mt:
        print(f"\n  --- Tile-level Cache: Multi-threaded Reads (misaligned) ---")
        mt_rows = []
        for label, r in [("cuslide", cuslide_results), ("cuslide2", cuslide2_results)]:
            if "cache_mt_ms" in r:
                seq_ms = r.get("cache_misaligned_ms", 0)
                mt_speedup = seq_ms / r["cache_mt_ms"] if r["cache_mt_ms"] > 0 else 0
                mt_rows.append({
                    "Plugin": label,
                    "Sequential (ms)": f"{seq_ms:.2f}" if seq_ms else "N/A",
                    f"Threaded {r['cache_mt_workers']}T (ms)": f"{r['cache_mt_ms']:.2f}",
                    "Thread speedup": f"{mt_speedup:.2f}x" if seq_ms else "N/A",
                    "Hits/misses": f"{r['cache_mt_hits']}/{r['cache_mt_misses']}",
                    "Patches": r["cache_mt_n_patches"],
                })
        display(pd.DataFrame(mt_rows))

## Visualization

In [ ]:
n_images = len(all_results)
fig, axes = plt.subplots(n_images, 3, figsize=(20, 5 * n_images), squeeze=False)

for row_idx, (image_label, res) in enumerate(all_results.items()):
    cuslide_results = res["cuslide"]
    cuslide2_results = res["cuslide2"]

    # --- Column 1: Bar chart — latency comparison ---
    ax = axes[row_idx, 0]
    labels = []
    cs_means = []
    cs_stds = []
    cs2_means = []
    cs2_stds = []

    for bm in BENCH_NAMES:
        cs = cuslide_results.get(bm, {})
        cs2 = cuslide2_results.get(bm, {})
        if "error" in cs or "error" in cs2:
            continue
        labels.append(bm.replace("_", "\n"))
        cs_means.append(cs["mean_ms"])
        cs_stds.append(cs["std_ms"])
        cs2_means.append(cs2["mean_ms"])
        cs2_stds.append(cs2["std_ms"])

    x = np.arange(len(labels))
    width = 0.35

    ax.bar(x - width/2, cs_means, width, yerr=cs_stds, label="cuslide",
           color="#4c72b0", alpha=0.85, capsize=3)
    ax.bar(x + width/2, cs2_means, width, yerr=cs2_stds, label="cuslide2",
           color="#55a868", alpha=0.85, capsize=3)

    ax.set_ylabel("Latency (ms)")
    ax.set_title(f"{image_label}: Latency Comparison")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    # --- Column 2: Cache cold vs warm ---
    ax2 = axes[row_idx, 1]
    has_cs_cache = "cache_cold_ms" in cuslide_results
    has_cs2_cache = "cache_cold_ms" in cuslide2_results

    if has_cs_cache or has_cs2_cache:
        cache_labels = []
        cache_cold_vals = []
        cache_warm_vals = []
        cache_warm_errs = []

        if has_cs_cache:
            cache_labels.append("cuslide")
            cache_cold_vals.append(cuslide_results["cache_cold_ms"])
            cache_warm_vals.append(cuslide_results["cache_warm_mean_ms"])
            cache_warm_errs.append(cuslide_results["cache_warm_std_ms"])
        if has_cs2_cache:
            cache_labels.append("cuslide2")
            cache_cold_vals.append(cuslide2_results["cache_cold_ms"])
            cache_warm_vals.append(cuslide2_results["cache_warm_mean_ms"])
            cache_warm_errs.append(cuslide2_results["cache_warm_std_ms"])

        x_cache = np.arange(len(cache_labels))
        w_bar = 0.35
        ax2.bar(x_cache - w_bar/2, cache_cold_vals, w_bar, label="Cold (decode+insert)",
                color="#dd8452", alpha=0.85)
        ax2.bar(x_cache + w_bar/2, cache_warm_vals, w_bar, yerr=cache_warm_errs,
                label="Warm (cache hit)", color="#55a868", alpha=0.85, capsize=5)

        for i, (cold, warm) in enumerate(zip(cache_cold_vals, cache_warm_vals)):
            speedup = cold / warm if warm > 0 else 0
            ax2.text(i, max(cold, warm) + 0.5, f"{speedup:.1f}x",
                     ha="center", va="bottom", fontsize=9, fontweight="bold")

        ax2.set_ylabel("Latency (ms)")
        ax2.set_title(f"{image_label}: Cache Cold vs Warm")
        ax2.set_xticks(x_cache)
        ax2.set_xticklabels(cache_labels)
        ax2.legend()
        ax2.grid(axis="y", alpha=0.3)
    else:
        ax2.text(0.5, 0.5, "Cache data not available",
                 ha="center", va="center", transform=ax2.transAxes, fontsize=12)
        ax2.set_title(f"{image_label}: Cache Cold vs Warm")

    # --- Column 3: Aligned vs misaligned + multi-threaded ---
    ax3 = axes[row_idx, 2]
    has_align_data = ("cache_aligned_ms" in cuslide_results or
                      "cache_aligned_ms" in cuslide2_results)

    if has_align_data:
        bar_labels = []
        bar_vals = []
        bar_colors = []

        for label, r, base_color in [("cuslide", cuslide_results, "#4c72b0"),
                                      ("cuslide2", cuslide2_results, "#55a868")]:
            if "cache_aligned_ms" in r:
                bar_labels.append(f"{label}\naligned")
                bar_vals.append(r["cache_aligned_ms"])
                bar_colors.append(base_color)

                bar_labels.append(f"{label}\nmisaligned")
                bar_vals.append(r["cache_misaligned_ms"])
                bar_colors.append(base_color.replace("b0", "e0") if "4c72" in base_color
                                  else base_color.replace("68", "b8"))

            if "cache_mt_ms" in r:
                bar_labels.append(f"{label}\nMT {r['cache_mt_workers']}T")
                bar_vals.append(r["cache_mt_ms"])
                bar_colors.append("#dd8452" if "cuslide2" not in label else "#c44e52")

        x_align = np.arange(len(bar_labels))
        bars = ax3.bar(x_align, bar_vals, color=bar_colors, alpha=0.85)

        for i, (lbl, val) in enumerate(zip(bar_labels, bar_vals)):
            ax3.text(i, val + 0.3, f"{val:.1f}", ha="center", va="bottom", fontsize=7)

        ax3.set_ylabel("Total time (ms)")
        ax3.set_title(f"{image_label}: Cache — Aligned vs Misaligned vs MT")
        ax3.set_xticks(x_align)
        ax3.set_xticklabels(bar_labels, fontsize=7)
        ax3.grid(axis="y", alpha=0.3)
    else:
        ax3.text(0.5, 0.5, "Aligned/misaligned data not available",
                 ha="center", va="center", transform=ax3.transAxes, fontsize=12)
        ax3.set_title(f"{image_label}: Cache Aligned vs Misaligned")

plt.tight_layout()
plt.savefig("cuslide_vs_cuslide2_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: cuslide_vs_cuslide2_benchmark.png")

## Speedup Summary

In [ ]:
for image_label, res in all_results.items():
    cuslide_results = res["cuslide"]
    cuslide2_results = res["cuslide2"]

    print("=" * 60)
    print(f"  SPEEDUP SUMMARY: {image_label}")
    print("=" * 60)

    for bm in BENCH_NAMES:
        cs = cuslide_results.get(bm, {})
        cs2 = cuslide2_results.get(bm, {})
        if "error" in cs or "error" in cs2:
            print(f"  {bm:30s}  SKIPPED (error)")
            continue
        speedup = cs["mean_ms"] / cs2["mean_ms"] if cs2["mean_ms"] > 0 else 0
        indicator = "faster" if speedup > 1 else "slower"
        print(f"  {bm:30s}  {speedup:.2f}x ({indicator})")

    print(f"\n  {'--- Tile Cache ---':^60s}")
    for label, r in [("cuslide", cuslide_results), ("cuslide2", cuslide2_results)]:
        if "cache_cold_ms" in r:
            print(f"  {f'{label} cold->warm':30s}  "
                  f"{r['cache_speedup']:.2f}x")
        if "cache_aligned_ms" in r and "cache_misaligned_ms" in r:
            aligned_per = r["cache_aligned_ms"] / r["cache_aligned_n_patches"]
            misaligned_per = r["cache_misaligned_ms"] / r["cache_misaligned_n_patches"]
            ratio = misaligned_per / aligned_per if aligned_per > 0 else 0
            print(f"  {f'{label} aligned (per tile)':30s}  "
                  f"{aligned_per:.2f} ms  (hits={r['cache_aligned_hits']})")
            print(f"  {f'{label} misaligned (per tile)':30s}  "
                  f"{misaligned_per:.2f} ms  (hits={r['cache_misaligned_hits']})")
        if "cache_mt_ms" in r and "cache_misaligned_ms" in r:
            seq_ms = r["cache_misaligned_ms"]
            mt_ms = r["cache_mt_ms"]
            mt_workers = r["cache_mt_workers"]
            mt_speedup = seq_ms / mt_ms if mt_ms > 0 else 0
            mt_label = f"{label} MT {mt_workers}T vs seq"
            print(f"  {mt_label:30s}  "
                  f"{mt_speedup:.2f}x  (hits={r['cache_mt_hits']})")

    print("=" * 60)
    print()

## Raw Results (JSON)

In [ ]:
for image_label, res in all_results.items():
    print(f"\n{'='*40}")
    print(f"  {image_label}")
    print(f"{'='*40}")
    print("\n--- cuslide ---")
    print(json.dumps(res["cuslide"], indent=2))
    print("\n--- cuslide2 ---")
    print(json.dumps(res["cuslide2"], indent=2))